# Session 6 · Planning Complex Tasks: Events, Campaigns and Projects

Generative AI › Track 3 · Innovate with Generative AI

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-06_critical_path.py`

In [1]:
"""
Session 6 · Example 1 — Critical path of an event plan (CPM).

AI assistants are good at listing tasks for a big job, but they rarely
compute the schedule correctly. The Critical Path Method does:
  * earliest start (ES) of a task = latest earliest-finish of its prerequisites
  * the project length = largest earliest finish
  * latest start (LS) found by walking backwards; slack = LS - ES
  * tasks with zero slack form the CRITICAL PATH: any delay there delays everything.

Run:  python session-06_critical_path.py
"""
tasks = {  # id: (days, prerequisites, description)
    "A": (3, [],          "Define goals, budget, audience"),
    "B": (5, ["A"],       "Book venue and date"),
    "C": (4, ["A"],       "Invite speakers"),
    "D": (6, ["B", "C"],  "Design and print materials"),
    "E": (7, ["B"],       "Run promotion campaign"),
    "F": (2, ["C"],       "Confirm speaker travel"),
    "G": (2, ["D", "E", "F"], "Rehearsal and final checks"),
}

# Forward pass (dict order is already a valid topological order here)
ES, EF = {}, {}
for t, (d, pre, _) in tasks.items():
    ES[t] = max((EF[p] for p in pre), default=0)
    EF[t] = ES[t] + d
length = max(EF.values())

# Backward pass
LF, LS = {}, {}
for t in reversed(list(tasks)):
    succ = [s for s, (_, pre, _) in tasks.items() if t in pre]
    LF[t] = min((LS[s] for s in succ), default=length)
    LS[t] = LF[t] - tasks[t][0]

print(f"{'Task':4} {'Days':>4} {'ES':>3} {'EF':>3} {'LS':>3} {'Slack':>5}  Description")
for t, (d, _, desc) in tasks.items():
    slack = LS[t] - ES[t]
    mark = "  <- critical" if slack == 0 else ""
    print(f"{t:4} {d:4} {ES[t]:3} {EF[t]:3} {LS[t]:3} {slack:5}  {desc}{mark}")
critical = [t for t in tasks if LS[t] == ES[t]]
print(f"\nProject length: {length} days | Critical path: {' -> '.join(critical)}")

Task Days  ES  EF  LS Slack  Description
A       3   0   3   0     0  Define goals, budget, audience  <- critical
B       5   3   8   3     0  Book venue and date  <- critical
C       4   3   7   5     2  Invite speakers
D       6   8  14   9     1  Design and print materials
E       7   8  15   8     0  Run promotion campaign  <- critical
F       2   7   9  13     6  Confirm speaker travel
G       2  15  17  15     0  Rehearsal and final checks  <- critical

Project length: 17 days | Critical path: A -> B -> E -> G


### Example · `session-06_event_checklist.py`

In [2]:
"""
Session 6 · Example 3 — Back-schedule an event checklist from its date.

Planners work BACKWARDS from the event day: "posters must be printed 10 days
before", "venue booked 30 days before". Give an AI the list of tasks and
lead times, but let code compute the dates (AIs often miscount weekdays).
Tasks that fall on a Sunday are moved to the Saturday before.

Run:  python session-06_event_checklist.py
"""
from datetime import date, timedelta

EVENT = date(2026, 12, 12)   # Saturday 12 December 2026
tasks = [  # (days before event, task, owner)
    (42, "Agree goals, budget and theme", "Lead"),
    (35, "Book auditorium and AV equipment", "Logistics"),
    (28, "Confirm speakers and judges", "Lead"),
    (21, "Launch registration form and social campaign", "Marketing"),
    (14, "Print posters and badges", "Design"),
    (6,  "Send reminder email and agenda", "Marketing"),
    (2,  "Rehearsal and tech check", "Logistics"),
    (0,  "EVENT DAY", "Everyone"),
    (-3, "Send thank-you notes and feedback survey", "Lead"),
]

print(f"Event: {EVENT:%A %d %B %Y}\n")
print(f"{'Due':16}{'D-':>5}  {'Owner':10} Task")
for days, task, owner in sorted(tasks, reverse=True):
    due = EVENT - timedelta(days=days)
    moved = ""
    if due.weekday() == 6:              # Sunday -> Saturday before
        due -= timedelta(days=1)
        moved = "  (moved from Sunday)"
    print(f"{due:%a %d %b %Y}{days:>5}  {owner:10} {task}{moved}")

Event: Saturday 12 December 2026

Due                D-  Owner      Task
Sat 31 Oct 2026   42  Lead       Agree goals, budget and theme
Sat 07 Nov 2026   35  Logistics  Book auditorium and AV equipment
Sat 14 Nov 2026   28  Lead       Confirm speakers and judges
Sat 21 Nov 2026   21  Marketing  Launch registration form and social campaign
Sat 28 Nov 2026   14  Design     Print posters and badges
Sat 05 Dec 2026    6  Marketing  Send reminder email and agenda  (moved from Sunday)
Thu 10 Dec 2026    2  Logistics  Rehearsal and tech check
Sat 12 Dec 2026    0  Everyone   EVENT DAY
Tue 15 Dec 2026   -3  Lead       Send thank-you notes and feedback survey


### Example · `session-06_campaign_budget.py`

In [3]:
"""
Session 6 · Example 2 — Split a campaign budget and forecast results.

Paid social is priced per 1,000 impressions (CPM). From the budget and CPM
we can forecast impressions; a click-through rate (CTR) gives clicks; a
conversion rate gives sign-ups; and cost per acquisition (CPA) = spend / sign-ups.
The CPMs and rates below are ASSUMED planning numbers for a student campaign,
not real market data: replace them with figures from your own ad dashboard.

Run:  python session-06_campaign_budget.py
"""
BUDGET = 50_000  # rupees
channels = {  # share of budget, CPM (Rs), CTR, conversion rate
    "Instagram": (0.40, 180, 0.012, 0.05),
    "Facebook":  (0.25, 120, 0.009, 0.04),
    "YouTube":   (0.20, 250, 0.006, 0.03),
    "LinkedIn":  (0.15, 600, 0.008, 0.08),
}

assert abs(sum(c[0] for c in channels.values()) - 1) < 1e-9, "shares must add up to 100%"
rows, tot = [], {"spend": 0, "imp": 0, "clicks": 0, "signups": 0}
print(f"{'Channel':10}{'Spend':>8}{'Impr.':>10}{'Clicks':>8}{'Sign-ups':>9}{'CPA':>8}")
for name, (share, cpm, ctr, cvr) in channels.items():
    spend = BUDGET * share
    imp = spend / cpm * 1000
    clicks = imp * ctr
    signups = clicks * cvr
    cpa = spend / signups
    print(f"{name:10}{spend:8,.0f}{imp:10,.0f}{clicks:8,.0f}{signups:9,.0f}{cpa:8,.0f}")
    for k, v in zip(tot, (spend, imp, clicks, signups)):
        tot[k] += v
print(f"{'TOTAL':10}{tot['spend']:8,.0f}{tot['imp']:10,.0f}{tot['clicks']:8,.0f}"
      f"{tot['signups']:9,.0f}{tot['spend']/tot['signups']:8,.0f}")

Channel      Spend     Impr.  Clicks Sign-ups     CPA
Instagram   20,000   111,111   1,333       67     300
Facebook    12,500   104,167     938       38     333
YouTube     10,000    40,000     240        7   1,389
LinkedIn     7,500    12,500     100        8     938
TOTAL       50,000   267,778   2,611      119     419


### Example · `session-06_project_campaign_planner.py`

In [4]:
"""
Session 6 · Mini project (full solution) — Campaign planner with a Gantt chart.

Real-life problem: plan a 4-week admissions campaign. Inputs: a launch date,
a list of workstreams with durations and dependencies. Outputs:
  * a dated schedule (computed by the critical path method),
  * a weekly content calendar (which post goes out on which day),
  * a Mermaid gantt block you can paste into any Markdown page,
  * a Gantt chart PNG made with matplotlib.

Run:  python session-06_project_campaign_planner.py
Creates: session-06_gantt.png
"""
from datetime import date, timedelta

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

START = date(2026, 11, 2)   # Monday
work = {  # id: (name, days, prerequisites)
    "res": ("Audience research with AI", 3, []),
    "msg": ("Key messages and slogan", 2, ["res"]),
    "vis": ("Visual kit: posters, templates", 5, ["msg"]),
    "vid": ("Promo video and captions", 6, ["msg"]),
    "web": ("Landing page copy", 3, ["msg"]),
    "run": ("Campaign live: posts and ads", 14, ["vis", "vid", "web"]),
    "rep": ("Results report", 2, ["run"]),
}

es, ef = {}, {}
for k, (_, d, pre) in work.items():
    es[k] = max((ef[p] for p in pre), default=0)
    ef[k] = es[k] + d
end = max(ef.values())

# backward pass: latest start; zero slack = critical
ls = {}
for k in reversed(list(work)):
    succ = [s for s, (_, _, pre) in work.items() if k in pre]
    lf = min((ls[s] for s in succ), default=end)
    ls[k] = lf - work[k][1]
critical = [k for k in work if ls[k] == es[k]]

print("SCHEDULE")
for k, (name, d, _) in work.items():
    s, e = START + timedelta(es[k]), START + timedelta(ef[k] - 1)
    print(f"  {name:32} {s:%d %b} -> {e:%d %b}  ({d} d)")
print(f"  Finish: {START + timedelta(end - 1):%a %d %b %Y} ({end} days)")
print(f"  Critical path: {' -> '.join(work[k][0] for k in critical)}")

print("\nCONTENT CALENDAR (campaign-live window)")
themes = ["Why study CS here", "Student story", "Lab tour reel", "Scholarship facts", "Q&A live"]
days = {0: "Mon", 2: "Wed", 4: "Fri"}
live = START + timedelta(es["run"])
n = 0
for offset in range(work["run"][1]):
    d = live + timedelta(offset)
    if d.weekday() in days:
        print(f"  {d:%a %d %b}: {themes[n % len(themes)]}")
        n += 1

print("\nMERMAID GANTT")
print("gantt\n    dateFormat YYYY-MM-DD\n    title Admissions campaign")
for k, (name, d, _) in work.items():
    print(f"    {name} :{k}, {START + timedelta(es[k])}, {d}d")

fig, ax = plt.subplots(figsize=(8, 3.5))
for i, (k, (name, d, _)) in enumerate(work.items()):
    ax.barh(i, d, left=es[k], color="#FF6B1A" if k in critical else "#A78BFA")
ax.set_yticks(range(len(work)))
ax.set_yticklabels([v[0] for v in work.values()], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel(f"Days from {START:%d %b %Y} (orange = critical path)")
ax.set_title("Admissions campaign plan")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig("session-06_gantt.png", dpi=80)
print("\nSaved -> session-06_gantt.png")

SCHEDULE
  Audience research with AI        02 Nov -> 04 Nov  (3 d)
  Key messages and slogan          05 Nov -> 06 Nov  (2 d)
  Visual kit: posters, templates   07 Nov -> 11 Nov  (5 d)
  Promo video and captions         07 Nov -> 12 Nov  (6 d)
  Landing page copy                07 Nov -> 09 Nov  (3 d)
  Campaign live: posts and ads     13 Nov -> 26 Nov  (14 d)
  Results report                   27 Nov -> 28 Nov  (2 d)
  Finish: Sat 28 Nov 2026 (27 days)
  Critical path: Audience research with AI -> Key messages and slogan -> Promo video and captions -> Campaign live: posts and ads -> Results report

CONTENT CALENDAR (campaign-live window)
  Fri 13 Nov: Why study CS here
  Mon 16 Nov: Student story
  Wed 18 Nov: Lab tour reel
  Fri 20 Nov: Scholarship facts
  Mon 23 Nov: Q&A live
  Wed 25 Nov: Why study CS here

MERMAID GANTT
gantt
    dateFormat YYYY-MM-DD
    title Admissions campaign
    Audience research with AI :res, 2026-11-02, 3d
    Key messages and slogan :msg, 2026-11-05, 2d
